# Calibration and robustness

**Research objective.** Present saved calibration, modality sensitivity, and raw-image robustness evidence with scope boundaries.

All results are read from local FNEWS artifacts. `SOURCE_TRUE` and `SOURCE_NON_TRUE` are Fakeddit source-taxonomy labels, not independently verified factuality labels. The held-out test partition is not opened by these notebooks except where a prior frozen metric is explicitly quoted.

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd()
if not (ROOT / "data/research_dataset").exists():
    ROOT = Path("/Users/aka_ank/Desktop/FNEWS")
DATA = ROOT / "data/research_dataset"
REPORTS = ROOT / "reports"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13, "axes.labelsize": 11})

def paired(split):
    frame = pd.read_parquet(DATA / "manifests" / f"{split}_manifest.parquet")
    return frame.loc[(frame.source_dataset == "Fakeddit") & (frame.download_status == "verified_reused_or_processed")].copy().reset_index(drop=True)

def canonical_source_labels(raw):
    """Fakeddit raw 1=true; return 0=SOURCE_TRUE, 1=SOURCE_NON_TRUE."""
    return 1 - np.asarray(raw, dtype=int)

def source_names(values):
    return np.where(np.asarray(values) == 0, "SOURCE_TRUE", "SOURCE_NON_TRUE")

def show_frame(frame, rows=10):
    display(frame.head(rows))


## Calibration

Temperature scaling maps logits to probabilities using a learned positive temperature. ECE measures calibration gap; Brier score measures squared probability error. Calibration does not establish factuality or repair domain shift.

In [2]:
cal=json.loads((REPORTS/'phase2/calibration/metrics.json').read_text());display(pd.json_normalize(cal));print('Saved temperature:',json.loads((REPORTS/'phase2/calibration/temperature.json').read_text()).get('temperature'))

,temperature,protocol.split,protocol.calibration_samples,protocol.assessment_samples,protocol.seed,protocol.heldout_test_accessed,assessment_raw.ece_10,assessment_raw.brier,assessment_raw.negative_log_likelihood,assessment_calibrated.ece_10,assessment_calibrated.brier,assessment_calibrated.negative_log_likelihood,calibration_raw.ece_10,calibration_raw.brier,calibration_raw.negative_log_likelihood,calibration_calibrated.ece_10,calibration_calibrated.brier,calibration_calibrated.negative_log_likelihood
0,2.532525,Fakeddit validation only,466,467,42,False,0.088905,0.101671,0.386519,0.037953,0.089823,0.287592,0.085325,0.111534,0.48317,0.052858,0.10446,0.337817


Saved temperature: 2.532524824142456


The saved assessment used 467 validation examples: ECE improved from 0.0889 to 0.0380 and Brier score from 0.1017 to 0.0898. This is a split-local calibration assessment.

In [3]:
ablation=json.loads((REPORTS/'phase2/modality_ablation/metrics.json').read_text());display(pd.DataFrame(ablation['conditions']).T)

,accuracy,macro_f1,prediction_change_rate_vs_paired,mean_absolute_probability_shift_vs_paired,gate_mean_text_weight,gate_std_text_weight,image_projection_norm_mean,text_projection_norm_mean,class_prediction_change_rate,bootstrap_macro_f1_delta_vs_shuffled_image_95ci,bootstrap_macro_f1_delta_vs_shuffled_text_95ci
paired,0.874598,0.8707,0.0,0.0,0.519165,0.190218,9.135094,9.668982,"{'0': 0.0, '1': 0.0}","[0.1300029233904467, 0.19437877376788446]","[0.19831487769257267, 0.263212666993924]"
shuffled_image,0.726688,0.707665,0.257235,0.249307,0.519329,0.191306,9.135095,9.668982,"{'0': 0.17446043165467626, '1': 0.379310344827...",NaN,NaN
shuffled_text,0.65702,0.638162,0.303323,0.298586,0.519188,0.191578,9.135094,9.668981,"{'0': 0.23741007194244604, '1': 0.400530503978...",NaN,NaN


## Robustness

Frozen modality shuffling changes macro-F1 from 87.07% paired to 70.77% with shuffled images and 63.82% with shuffled text. The raw-image sweep is a 32-pair, 19-condition subset experiment—not a full-validation robustness estimate. Severe blur sigma 3 changed subset macro-F1 from 87.50% to 77.03%.

In [4]:
paths=list((REPORTS/'phase2').rglob('*robust*.json'));print('Available robustness JSON:',[p.name for p in paths]);
for path in paths:
 print('\n',path);display(pd.json_normalize(json.loads(path.read_text())).head())

Available robustness JSON: []
